In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import glob, os, shutil, cv2, json, sys
from tqdm import tqdm
from pathlib import Path
import papermill as pm

sys.path.append('../..')
from Dataset.index import Normalization, getFiles

In [ ]:
DATASET01 = {'path': '../dataset_74', 'n_images': 110}
DATASET02 = {'path': '../dataset_wu', 'n_images': None}

In [3]:
BASE_PATH = f'../Dataset/{Path().resolve().name}'
BASE_PATH

'../Dataset/dataset_74_wu'

In [4]:
OPTIONS = json.loads(open('../../Task/info.json', 'r').read())
OPTIONS['dataset'] = BASE_PATH.split('/')[-1].strip()
SCALING = OPTIONS.get('scaling', Normalization.DEFAULT)
OPTIONS

{'network': 'fault_edge_former',
 'dataset': 'dataset_74_wu',
 'img_size': None,
 'lr': 0.0001,
 'loss': 'dice_focal',
 'batch_size': 2,
 'scheduler': 'plateau',
 'dropout': 0.05,
 'num_filters': 36,
 'ema': True,
 'n_trials': 1,
 'trial': 0}

### FONTES

- O `dataset_74_wu` não tem `images/` próprios: junta os volumes do `DATASET01` e do `DATASET02` como os `Format` deles gravaram, os `n_images` primeiros de cada um (`None` pega todos).
- A ordem importa: o split do `1 - Model` sorteia pela posição da linha. `DATASET01` = 110 do `dataset_74` e depois `DATASET02` = os 220 do `dataset_wu` reproduz byte a byte o `DataBase.csv` dos modelos salvos; com o wu na frente as contagens batem, mas os 15 volumes de teste caem todos no treino do `model_5`.
- A fonte cujo `DataBase.csv` não está no `scaling` da rodada tem o `Format` dela rodado aqui por papermill, como o `Task/index.py` faz, com o log em `Task/logs/Format_<fonte>_out.ipynb`.
- O `info.json` só é regravado depois, porque o `Format` de cada fonte o reescreve com o nome dela.
- No `'percentile'` cada fonte fica com o p01/p99 dela: as duas saem quase na mesma escala (±2,4 no `dataset_74`, ±2,7 no `dataset_wu`), e o p01/p99 da união mudaria o desvio em [0, 1] só de 0,20 para 0,19.

In [ ]:
# RODA O Format DE UMA FONTE; O SystemExit É O FIM NORMAL DELE QUANDO A RODADA NÃO TEM TILES
def execute(path):
    output = os.path.abspath(f'../../Task/logs/Format_{Path(path).parent.name}_out.ipynb')

    try:
        pm.execute_notebook(path, output, kernel_name='python3', log_output=True, progress_bar=True, cwd=str(Path(path).parent))
    except pm.PapermillExecutionError as error:
        if error.ename != 'SystemExit':
            raise


for source in (DATASET01, DATASET02):
    database = f"{source['path']}/DataBase.csv"

    if not os.path.exists(database) or Normalization.read(database) != SCALING:
        execute(f"{source['path']}/Format.ipynb")

    print(f"{source['path']}: scaling={Normalization.read(database)!r}")

In [5]:
with open('../../Task/info.json', 'w', encoding='utf-8') as file:
    json.dump(OPTIONS, file, ensure_ascii=False, indent=4) 

In [ ]:
d1_images = getFiles(f"{DATASET01['path']}/images", limit=DATASET01['n_images'])
d1_masks  = getFiles(f"{DATASET01['path']}/masks", limit=DATASET01['n_images'])

print(len(d1_images))
d1_images[:5]

220


['/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0000.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0001.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0002.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0003.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0004.npy']

In [7]:
d2_images = getFiles(f"{DATASET02['path']}/images", limit=DATASET02['n_images'])
d2_masks  = getFiles(f"{DATASET02['path']}/masks", limit=DATASET02['n_images'])
print(len(d2_images))
d2_images[:5]

220


['/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/0.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/1.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/10.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/100.npy',
 '/home/grva-mint/Projects/Falhas/Dataset/dataset_wu/images/101.npy']

In [8]:
df = pd.DataFrame()
df['img_path']  = d1_images + d2_images
df['mask_path'] = d1_masks  + d2_masks
df

,img_path,mask_path
0,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...
435,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
436,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
437,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
438,/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...


In [9]:
info = []

for img_path, mask_path in tqdm(zip(df.img_path, df.mask_path)):
    img  = np.load(img_path)
    mask = np.load(mask_path)
    data = dict()

    data['id'] = Path(img_path).name
    data['img_min']  = np.min(img)
    data['img_max']  = np.max(img)
    data['img_mean'] = np.mean(img)
    data['img_std']  = np.std(img)

    data['msk_min'] = np.min(mask)
    data['msk_max'] = np.max(mask)
    data['shape']   = img.shape
    data['img_path']  = img_path
    data['mask_path'] = mask_path
    data['scaling'] = SCALING
    info.append(data)


df = pd.DataFrame(info)
print(df.img_path.iloc[2])
df

440it [00:04, 105.92it/s]

/home/grva-mint/Projects/Falhas/Dataset/dataset_74/images/img_0002.npy


,id,img_min,img_max,img_mean,img_std,msk_min,msk_max,shape,img_path,mask_path
0,img_0000.npy,0.0,1.0,0.498273,0.206325,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,img_0001.npy,0.0,1.0,0.498135,0.204917,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,img_0002.npy,0.0,1.0,0.497561,0.203653,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,img_0003.npy,0.0,1.0,0.499910,0.199585,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,img_0004.npy,0.0,1.0,0.499684,0.172918,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...,...,...,...,...,...,...,...,...
435,95.npy,0.0,1.0,0.495437,0.192614,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
436,96.npy,0.0,1.0,0.495380,0.201204,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
437,97.npy,0.0,1.0,0.495210,0.186244,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
438,98.npy,0.0,1.0,0.494706,0.201227,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...


In [10]:
df.to_csv('DataBase.csv', index=None)
df

,id,img_min,img_max,img_mean,img_std,msk_min,msk_max,shape,img_path,mask_path
0,img_0000.npy,0.0,1.0,0.498273,0.206325,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
1,img_0001.npy,0.0,1.0,0.498135,0.204917,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
2,img_0002.npy,0.0,1.0,0.497561,0.203653,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
3,img_0003.npy,0.0,1.0,0.499910,0.199585,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
4,img_0004.npy,0.0,1.0,0.499684,0.172918,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
...,...,...,...,...,...,...,...,...,...,...
435,95.npy,0.0,1.0,0.495437,0.192614,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
436,96.npy,0.0,1.0,0.495380,0.201204,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
437,97.npy,0.0,1.0,0.495210,0.186244,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
438,98.npy,0.0,1.0,0.494706,0.201227,0.0,1.0,"(128, 128, 128)",/home/grva-mint/Projects/Falhas/Dataset/datase...,/home/grva-mint/Projects/Falhas/Dataset/datase...
